Analyzed channels.ipynb:L1-423

这份关于 [channels.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_convolutional-neural-networks/channels.ipynb)（多输入多输出通道）的内容总结如下：

---

### 一、核心背景与动机
* **从二维到三维/四维张量**：实际图像（如 RGB 3 通道）和深度卷积特征图都是多通道的，输入形状为 `c_i × h × w`。
* **深层网络设计模式**：随着网络层数加深，通常会**降低空间分辨率**（减小 `h, w`），同时**增加通道维度**（增大 `c_o`），以便捕捉更丰富、更抽象的高阶特征响应。

---

### 二、多输入通道（Multiple Input Channels）
1. **张量形状**：
   * 输入数据 `X`：`c_i × h × w`
   * 卷积核 `K`：`c_i × k_h × k_w`
   * 输出数据 `Y`：`h_out × w_out`（单通道）
2. **计算过程**：
   * 对每个输入通道分别进行二维互相关运算；
   * 将所有输入通道的二维互相关结果**按元素相加（通道求和）**，得到单个二维输出矩阵。
3. **实现核心代码**：
   ```python
   def corr2d_multi_in(X, K):
       # 遍历通道维度，将每个通道的互相关结果相加
       return sum(d2l.corr2d(x, k) for x, k in zip(X, K))
   ```

---

### 三、多输出通道（Multiple Output Channels）
1. **张量形状**：
   * 输入数据 `X`：`c_i × h × w`
   * 卷积核 `K`：`c_o × c_i × k_h × k_w`
   * 输出数据 `Y`：`c_o × h_out × w_out`
2. **计算过程**：
   * 为每个输出通道分配一个形状为 `c_i × k_h × k_w` 的 3D 卷积核；
   * 每个输出通道独立对整个输入进行多输入单输出卷积计算；
   * 最终在第 0 维（输出通道维）拼接（`torch.stack`）成 3D 张量。
3. **实现核心代码**：
   ```python
   def corr2d_multi_in_out(X, K):
       # 迭代 K 的第 0 维（输出通道），依次计算并堆叠
       return torch.stack([corr2d_multi_in(X, k) for k in K], 0)
   ```

---

### 四、1×1 卷积层（1×1 Convolution）
1. **本质与特性**：
   * 窗口大小 `k_h = k_w = 1`，完全**失去了空间局部相关性的提取能力**；
   * 计算完全发生在**通道维度**上，输出的每个像素是输入对应位置所有通道特征的线性组合；
   * **等价于在每个像素位置独立应用全连接层 (MLP)**，且所有像素位置共享同一组权重（权重形状为 `c_o × c_i`）。
2. **实际工程作用**：
   * **升维/降维**：灵活调节网络中特征图的通道数 `c_o`，从而控制参数量与计算复杂度（如 ResNet、Inception 瓶颈结构）；
   * **跨通道信息交互**：实现不同通道之间特征的融合与重组。
3. **全连接矩阵乘法高效实现**：
   ```python
   def corr2d_multi_in_out_1x1(X, K):
       c_i, h, w = X.shape
       c_o = K.shape[0]
       # 展平空间维度：X 变为 (c_i, h * w)，K 变为 (c_o, c_i)
       X = X.reshape((c_i, h * w))
       K = K.reshape((c_o, c_i))
       # 矩阵乘法：(c_o, c_i) @ (c_i, h * w) -> (c_o, h * w)
       Y = torch.matmul(K, X)
       return Y.reshape((c_o, h, w))
   ```

---

### 五、总结要点速查
* **输入通道求和**：融合所有输入通道的信息，输出维度坍缩掉 `c_i`。
* **输出通道堆叠**：通过增加不同核组的数量 `c_o`，生成不同视角的特征表示。
* **1×1 卷积**：跨通道特征线性变换与降维利器，数学等价于全连接层。

In [1]:
import torch
from d2l import torch as d2l

In [2]:
def corr2d_muti_in(X,K):
    return sum(d2l.corr2d(x, k) for x,k in zip(X, K))

In [9]:
X= torch.tensor(range(18)).reshape((2,3,3))
K =torch.tensor(range(8)).reshape(2,2,2)

corr2d_muti_in(X, K)

tensor([[268., 296.],
        [352., 380.]])

In [10]:
def corr2d_muti_in_out(X, K):
    return torch.stack([corr2d_muti_in(X, k) for k in K], 0 )

In [11]:
K = torch.stack((K, K+1, K+2), 0)
K.shape

torch.Size([3, 2, 2, 2])

In [12]:
corr2d_muti_in_out(X, K)

tensor([[[268., 296.],
         [352., 380.]],

        [[320., 356.],
         [428., 464.]],

        [[372., 416.],
         [504., 548.]]])

In [13]:
def corr2d_muti_in_out_1x1(X, K):
    ci, w, h = X.shape
    c0 = K.shape[0]
    X = X.reshape((ci, w*h))
    K = K.reshape((c0, ci))
    Y = torch.matmul(K, X)
    return Y.reshape((c0, h, w))


In [14]:
X = torch.normal(0,1, (3,3,3))
K = torch.normal(0,1, (2,3,1,1))

In [16]:
Y1 = corr2d_muti_in_out_1x1(X, K)
Y2 = corr2d_muti_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

以下是针对 [channels.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_convolutional-neural-networks/channels.ipynb) 课后练习题的详细解答：

---

### 练习 1
**假设我们有两个卷积核，大小分别为 `k_1` 和 `k_2`（中间没有非线性激活函数）。**
1. **证明运算可以用单次卷积来表示。**
2. **这个等效的单个卷积核的维数是多少呢？**
3. **反之亦然吗？**

#### 解答：
1. **单次卷积表示的证明**：
   * 设输入为 `X`，两个连续卷积核为 `K_1` 和 `K_2`。连续执行两次卷积输出为：
     ```text
     Y = (X * K_1) * K_2
     ```
   * 离散卷积满足**结合律**：`(X * K_1) * K_2 = X * (K_1 * K_2)`。
   * 从信号与系统的线性时不变（LTI）算子角度看，两个连续的线性移位等变算子的复合依然是线性移位等变的。
   * 因此，令等效卷积核 `K = K_1 * K_2`，则 `Y = X * K`，即连续两次线性卷积完全可以用一个单次卷积来等效表示。

2. **等效单个卷积核的尺寸/维数**：
   * 若 `K_1` 大小为 `k_1 × k_1`，`K_2` 大小为 `k_2 × k_2`，则两个核做离散全卷积后，等效卷积核 `K` 的空间尺寸为：
     ```text
     (k_1 + k_2 - 1) × (k_1 + k_2 - 1)
     ```
   * 例如：两个连续的 `3×3` 卷积核复合后，感受野等效于一个 `(3 + 3 - 1) × (3 + 3 - 1) = 5×5` 的单次卷积核。
   * 若考虑通道维度，设输入通道为 `c_in`、中间通道为 `c_mid`、输出通道为 `c_out`，则等效核的整体形状为 `c_out × c_in × (k_1 + k_2 - 1) × (k_1 + k_2 - 1)`。

3. **反之亦然吗（任意大的卷积核能否无损分解为两个小卷积核）？**
   * **反之通常不成立**。
   * **参数自由度分析**：一个 `5×5` 的单通道卷积核拥有 `25` 个独立自由参数；而两个 `3×3` 卷积核的参数量仅为 `3×3 + 3×3 = 18`。由于参数自由度不足（`18 < 25`），任意一个 `5×5` 卷积核无法完全被分解为两个 `3×3` 卷积核的级联。
   * **代数因式分解角度**：二维多元多项式通常是**不可约（irreducible）**的，不像一维复多项式那样总能因式分解为一次多项式的乘积。

---

### 练习 2
**假设输入为 `c_i × h × w`，卷积核大小为 `c_o × c_i × k_h × k_w`，填充为 `(p_h, p_w)`，步幅为 `(s_h, s_w)`。**
1. **前向传播的计算成本（乘法和加法）是多少？**
2. **内存占用是多少？**
3. **反向传播的内存占用是多少？**
4. **反向传播的计算成本是多少？**

#### 解答：
首先计算输出张量的空间尺寸 `(h_out, w_out)`：
```text
h_out = ⌊(h - k_h + p_h) / s_h⌋ + 1
w_out = ⌊(w - k_w + p_w) / s_w⌋ + 1
```
输出总元素个数为 `c_o × h_out × w_out`。

1. **前向传播的计算成本**：
   * 输出的每个元素需要对 `c_i` 个通道、每个通道 `k_h × k_w` 个窗口元素进行逐点乘积后求和。
   * **乘法次数**：
     ```text
     Multiplications = c_o × h_out × w_out × c_i × k_h × k_w
     ```
   * **加法次数**：
     * 无偏置项时：`c_o × h_out × w_out × (c_i × k_h × k_w - 1)`
     * 含有偏置项时：每个输出元素额外加一次 bias，即 `c_o × h_out × w_out × c_i × k_h × k_w`
   * **总 FLOPs（浮点运算次数）** 约为：
     ```text
     FLOPs ≈ 2 × c_o × c_i × k_h × k_w × h_out × w_out
     ```

2. **前向传播的内存占用**：
   * **模型参数内存**：权重张量 `c_o × c_i × k_h × k_w`（若有偏置则额外 `+ c_o`）。
   * **激活值内存（特征图）**：
     * 输入特征图：`c_i × h × w`
     * 输出特征图：`c_o × h_out × w_out`（训练阶段需保存在内存中供反向传播求导使用）。
   * （若使用 `im2col` 矩阵乘法实现，前向还会产生临时的展开缓冲区，大小为 `(c_i · k_h · k_w) × (h_out · w_out)`）。

3. **反向传播的内存占用**：
   * **梯度张量存储**：
     * 损失对输入的梯度 `∂L/∂X`：`c_i × h × w`
     * 损失对卷积核的梯度 `∂L/∂W`：`c_o × c_i × k_h × k_w`（偏置梯度 `c_o`）
     * 接收到的反传输出梯度 `∂L/∂Y`：`c_o × h_out × w_out`
   * **前向缓存依赖**：反向计算 `∂L/∂W` 时必须依赖前向缓存的输入 `X`（`c_i × h × w`）。

4. **反向传播的计算成本**：
   * 反向传播包含两部分核心计算：
     1. 计算对权重的梯度 `∂L/∂W = (∂L/∂Y) * X`；
     2. 计算对输入的梯度 `∂L/∂X = (∂L/∂Y) * W`（相当于带转置/翻转核的卷积）。
   * 每部分的运算量均与前向传播相当，因此反向传播的总计算成本通常是前向传播的 **2 倍** 左右（总 FLOPs 约为前向的 2 倍，乘法与加法次数总计约为前向的 2 倍）。

---

### 练习 3
**如果我们将输入通道 `c_i` 和输出通道 `c_o` 的数量加倍，计算数量会增加多少？如果我们把填充数量翻一番会怎么样？**

#### 解答：
1. **通道数翻倍**：
   * 卷积计算复杂度公式中包含乘积因子 `c_i · c_o`。
   * 若 `c_i → 2c_i` 且 `c_o → 2c_o`，则乘积项变为 `2 · 2 = 4` 倍。
   * **计算量将增加为原来的 4 倍（增加了 300%）**。
2. **填充数量翻倍**：
   * 填充 `p_h → 2p_h`，`p_w → 2p_w` 不改变卷积核和通道参数，但会使得输出分辨率 `h_out, w_out` 增大：
     ```text
     h_out_new = ⌊(h - k_h + 2p_h) / s_h⌋ + 1
     w_out_new = ⌊(w - k_w + 2p_w) / s_w⌋ + 1
     ```
   * 计算量正比于 `h_out × w_out`，计算量增加的倍数为：
     ```text
     增加比例 = (h_out_new × w_out_new) / (h_out_old × w_out_old)
     ```
   * 在特征图尺寸远大于 padding 时（`h >> p_h`），增幅较微弱；在深层特征图较小、padding 占比较大时增幅较明显。

---

### 练习 4
**如果卷积核的高度和宽度是 `k_h = k_w = 1`，前向传播的计算复杂度是多少？**

#### 解答：
* 当 `k_h = k_w = 1` 时，每个位置的计算退化为纯通道维度的线性变换（矩阵乘法）。
* 此时通常 `p_h = p_w = 0`，若步幅为 1，则 `h_out = h, w_out = w`。
* **乘法次数**：`c_o · c_i · h_out · w_out`
* **加法次数**：`c_o · (c_i - 1) · h_out · w_out`（含 bias 时为 `c_o · c_i · h_out · w_out`）
* **大 O 渐近复杂度**：
  ```text
  O(c_i · c_o · h_out · w_out)
  ```
  若步幅为 1，即为 `O(c_i · c_o · h · w)`。

---

### 练习 5
**本节最后一个示例中的变量 `Y1` 和 `Y2` 是否完全相同？为什么？**

#### 解答：
* **数学逻辑上完全相同，但在浮点数值上通常不逐位相同（not bit-wise identical）**。
* **原因分析**：
  1. `Y1` 是使用 `corr2d_multi_in_out_1x1` 计算的，底层直接调用高度优化的底层矩阵乘法算子 `torch.matmul`（通常基于 BLAS/cuBLAS，采用块矩阵相乘、SIMD 向量化指令或分层归约求和）。
  2. `Y2` 是使用 `corr2d_multi_in_out` 计算的，它通过 Python 的多层循环和 `sum(...)` 沿通道依次进行累加。
  3. **计算机浮点数加法不满足结合律**（即在有限浮点精度下，`(a + b) + c` 不严格等于 `a + (b + c)`）。不同的累加次序会导致微小的截断/舍入误差（floating-point roundoff error）。
  4. 因此两者存在约 `1e-7` 量级的数值偏差，代码中使用 `torch.abs(Y1 - Y2).sum() < 1e-6` 容差断言，而无法使用直接严格相等的 `torch.equal(Y1, Y2)`。

---

### 练习 6
**当卷积窗口不是 `1×1` 时，如何使用矩阵乘法实现卷积？**

#### 解答：
这就是现代深度学习计算库中标准的 **`im2col`（Image to Column）** 算法（在 PyTorch 中对应 [`torch.nn.functional.unfold`](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_convolutional-neural-networks/channels.ipynb)）。具体分为以下 4 步：

```text
步骤示意：
  输入 X: (c_i, h, w)  ──[im2col / unfold]──>  X_col: (c_i · k_h · k_w,  h_out · w_out)
  权重 K: (c_o, c_i, k_h, k_w)  ──[reshape]──>  W_row: (c_o,  c_i · k_h · k_w)
  
  矩阵乘法:
  Y_col = W_row @ X_col  ──>  形状为 (c_o,  h_out · w_out)
  
  重塑还原:
  Y_col  ──[reshape]──>  Y: (c_o, h_out, w_out)
```

1. **输入展开（im2col / Unfold）**：
   * 将输入图像中每一个滑动窗口对应的感受野小立方体（大小为 `c_i × k_h × k_w`）展平为一列。
   * 滑动窗口共可滑动 `h_out × w_out` 次，把所有展开的列并排摆放，拼成一个大矩阵 `X_col`，形状为：
     `(c_i · k_h · k_w) × (h_out · w_out)`。
2. **卷积核权重重塑**：
   * 将四维卷积核 `(c_o, c_i, k_h, k_w)` 直接展平为二维矩阵 `W_row`，形状为：
     `c_o × (c_i · k_h · k_w)`。
3. **调用通用矩阵乘法（GEMM）**：
   * 执行单次标准矩阵乘法：
     ```text
     Y_col = torch.matmul(W_row, X_col)
     ```
   * 得到矩阵 `Y_col`，形状为 `c_o × (h_out · w_out)`。
4. **还原输出形状（Reshape / col2im）**：
   * 将 `Y_col` 重新 reshape 为三维/四维张量 `(c_o, h_out, w_out)`，即得到标准的卷积输出。

> **权衡与优缺点**：
> * **优点**：将任意尺寸的局部卷积统一转化为标准矩阵乘法（GEMM），能完全利用 GPU / TPU 的张量核心计算吞吐。
> * **代价**：滑动窗口之间存在像素重叠，`X_col` 中会存在大量冗余重复存储，产生额外的内存开销。